# ComfyUI on Google Colab — FLUX.1 Kontext (image) + LTX-Video (video)

Notebook setup ComfyUI chạy trên Colab Pro (T4 16GB / L4 24GB GPU), expose qua cloudflared HTTPS tunnel để BE local (FastAPI ở `e:\Python\DATN-_HaUI\backend`) gọi vào.

## Cách dùng

1. **Runtime → Change runtime type → GPU** (T4 hoặc L4).
2. **Lần đầu**: Run all. Tải ~30GB models vào Google Drive (persistent, chỉ lần đầu). Subsequent sessions sẽ skip download, chỉ copy Drive→/content (~5-7 phút).
3. **Bước 7** in ra `COMFY_BASE_URL=https://....trycloudflare.com` — paste vào `backend/.env`, restart BE bằng `run_dev.bat`.
4. **Đừng đóng tab Colab** trong khi dùng — cell heartbeat (Bước 8) chạy mãi để giữ session sống.
5. Stop bằng cách click nút stop ở cell heartbeat — tự cleanup tunnel + ComfyUI.

## Stack model

| File | Size | Vai trò |
|---|---|---|
| `flux1-kontext-dev-Q3_K_S.gguf` | 5.2GB | FLUX Kontext UNet — tạo/sửa ảnh (GGUF Q3) |
| `t5xxl_fp8_e4m3fn.safetensors` | 4.9GB | T5xxl encoder — DÙNG CHUNG cho FLUX và LTX |
| `clip_l.safetensors` | 235MB | CLIP-L encoder (FLUX) |
| `ae.safetensors` | 335MB | FLUX VAE |
| `ltxv-2b-0.9.8-distilled-fp8.safetensors` | 4.46GB | LTX-Video 2B — i2v cho T4 16GB |
| `ltxv-13b-0.9.8-distilled-fp8.safetensors` | 15.7GB | LTX-Video 13B — i2v cho L4 24GB (nét hơn) |

Tổng disk: ~31GB. Cần Drive ≥ 35GB trống.

## Cách BE chọn model video

BE query VRAM của GPU qua `/system_stats` trước mỗi lần tạo video:
- **VRAM ≥ 20GB** (L4/A100) → dùng **13B fp8** (chất lượng cao)
- **VRAM < 20GB** (T4) → fallback **2B fp8** (không OOM)

Cả 2 là checkpoint all-in-one (VAE bundled), cùng workflow `i2v_ltxv.json`, chỉ khác tên file. FLUX dùng `COMFY_BACKEND=flux_kontext`. T5 encoder dùng chung (load `type=ltxv`).

## Bước 1: Mount Google Drive

Drive dùng làm persistent storage. Mỗi session Colab `/content/` bị wipe, nhưng Drive thì giữ nguyên — model tải 1 lần, dùng mãi.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/comfyui_models')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Drive root: {DRIVE_ROOT}')

## Bước 2: Tải model FLUX Kontext stack vào Drive (chỉ chạy lần đầu)

Subsequent runs sẽ skip nếu file đã tồn tại trên Drive với size đúng. Speed download trong Colab ~100-300MB/s nên tổng ~1-3 phút lần đầu.

**Lần đầu cần setup HF token** cho file `ae.safetensors` (FLUX VAE từ repo gated, anonymous wget sẽ ra file 0 byte):

1. Truy cập https://huggingface.co/black-forest-labs/FLUX.1-schnell → kéo xuống cuối, click **"Agree and access repository"**.
2. Truy cập https://huggingface.co/settings/tokens → **Create new token** → scope = **Read**. Copy token (`hf_xxx...`).
3. Trong Colab sidebar trái, click icon **🔑** ("Secrets") → **+ Add new secret** → name = `HF_TOKEN`, value = paste token → bật toggle **"Notebook access"**.

Các file còn lại (gguf UNet + 2 file CLIP) tải từ public URL, không cần auth.

In [ ]:
import os
import shutil
from pathlib import Path

# Public files (no auth needed)
PUBLIC = [
    (
        'diffusion_models/flux1-kontext-dev-Q3_K_S.gguf',
        'https://huggingface.co/QuantStack/FLUX.1-Kontext-dev-GGUF/resolve/main/flux1-kontext-dev-Q3_K_S.gguf',
        5200,
    ),
    (
        'clip/t5xxl_fp8_e4m3fn.safetensors',
        'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors',
        4900,
    ),
    (
        'clip/clip_l.safetensors',
        'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors',
        230,
    ),
    # LTX-Video i2v (image-to-video). All-in-one fp8 checkpoints (UNet+VAE);
    # reuse the t5xxl_fp8 above as T5 encoder (type=ltxv). BE picks 2B vs 13B
    # by GPU VRAM at runtime: T4 16GB -> 2B, L4 24GB -> 13B (sharper).
    (
        'checkpoints/ltxv-2b-0.9.8-distilled-fp8.safetensors',
        'https://huggingface.co/Lightricks/LTX-Video/resolve/main/ltxv-2b-0.9.8-distilled-fp8.safetensors',
        4460,
    ),
    (
        'checkpoints/ltxv-13b-0.9.8-distilled-fp8.safetensors',
        'https://huggingface.co/Lightricks/LTX-Video/resolve/main/ltxv-13b-0.9.8-distilled-fp8.safetensors',
        15700,
    ),
]

# Gated files (HF token required — see markdown above)
GATED = [
    {
        'rel_path': 'vae/ae.safetensors',
        'repo_id': 'black-forest-labs/FLUX.1-schnell',
        'filename': 'ae.safetensors',
        'expected_mb': 320,
    },
]


def needs_download(target: Path, expected_mb: int) -> bool:
    if not target.exists():
        return True
    sz_mb = target.stat().st_size / (1024 * 1024)
    if sz_mb < expected_mb * 0.95:
        print(f'[BAD] {target.name} size {sz_mb:.0f}MB < expected ~{expected_mb}MB, re-downloading')
        target.unlink()
        return True
    return False


# Public files via wget (surface errors with --tries=2, no --quiet)
for rel_path, url, expected_mb in PUBLIC:
    target = DRIVE_ROOT / rel_path
    target.parent.mkdir(parents=True, exist_ok=True)
    if not needs_download(target, expected_mb):
        print(f'[SKIP] {rel_path} ({target.stat().st_size/(1024*1024):.0f}MB)')
        continue
    print(f'[DOWNLOAD public] {rel_path} (~{expected_mb}MB)')
    ret = os.system(f'wget --tries=2 -O "{target}" "{url}"')
    if ret != 0:
        print(f'[FAIL] wget exit={ret} for {url}')
        target.unlink(missing_ok=True)

# Gated files via HF auth
if any(needs_download(DRIVE_ROOT / g['rel_path'], g['expected_mb']) for g in GATED):
    try:
        from google.colab import userdata
        from huggingface_hub import hf_hub_download
        HF_TOKEN = userdata.get('HF_TOKEN')
        assert HF_TOKEN, 'empty'
    except Exception as e:
        raise RuntimeError(
            'HF_TOKEN secret missing or empty. Setup (3 bước, ~2 phút):\n'
            '1. https://huggingface.co/black-forest-labs/FLUX.1-schnell -> "Agree and access"\n'
            '2. https://huggingface.co/settings/tokens -> create token (Read scope)\n'
            '3. Colab sidebar -> Secrets (🔑) -> add HF_TOKEN, enable Notebook access\n'
        ) from e

    for g in GATED:
        target = DRIVE_ROOT / g['rel_path']
        target.parent.mkdir(parents=True, exist_ok=True)
        if not needs_download(target, g['expected_mb']):
            print(f"[SKIP] {g['rel_path']} ({target.stat().st_size/(1024*1024):.0f}MB)")
            continue
        print(f"[DOWNLOAD gated] {g['rel_path']} via HF auth (~{g['expected_mb']}MB)")
        cached = hf_hub_download(
            repo_id=g['repo_id'],
            filename=g['filename'],
            token=HF_TOKEN,
            cache_dir='/tmp/hf_cache',
        )
        shutil.copy2(cached, target)
else:
    for g in GATED:
        target = DRIVE_ROOT / g['rel_path']
        print(f"[SKIP] {g['rel_path']} ({target.stat().st_size/(1024*1024):.0f}MB)")

print('\nAll models on Drive.')

## Bước 3: Cài ComfyUI + custom node ComfyUI-GGUF

ComfyUI clone vào `/content/` (mất khi session reset, OK). GGUF custom node để load `.gguf` UNet.

In [ ]:
import os
import subprocess

if not os.path.exists('/content/ComfyUI'):
    subprocess.run(
        ['git', 'clone', '--depth', '1', 'https://github.com/comfyanonymous/ComfyUI', '/content/ComfyUI'],
        check=True,
    )
    print('[OK] ComfyUI cloned')
else:
    print('[SKIP] ComfyUI already cloned')

print('Installing ComfyUI requirements...')
subprocess.run(
    ['pip', 'install', '-q', '-r', '/content/ComfyUI/requirements.txt'],
    check=True,
)

GGUF_DIR = '/content/ComfyUI/custom_nodes/ComfyUI-GGUF'
if not os.path.exists(GGUF_DIR):
    subprocess.run(
        ['git', 'clone', '--depth', '1', 'https://github.com/city96/ComfyUI-GGUF', GGUF_DIR],
        check=True,
    )
    print('[OK] ComfyUI-GGUF cloned')
else:
    print('[SKIP] ComfyUI-GGUF already cloned')

subprocess.run(
    ['pip', 'install', '-q', '-r', f'{GGUF_DIR}/requirements.txt'],
    check=True,
)
print('[OK] All deps installed')

## Bước 4: Copy models từ Drive vào `/content` (~3-5 phút mỗi session)

ComfyUI mới dùng `mmap` để load model, nhưng Google Drive FUSE **không hỗ trợ mmap** → symlink Drive→`/content` sẽ raise `ModelMMAP allocation failed` khi sample. Phải copy thật ~10.7GB sang `/content/ComfyUI/models/` (local SSD Colab) cho mmap chạy được.

Cell auto-skip nếu file đã copy (so size). Subsequent re-run trong cùng session sẽ skip ngay.

In [ ]:
import shutil
from pathlib import Path

COMFY_MODELS = Path('/content/ComfyUI/models')

# Detect GPU VRAM so we copy ONLY the LTX checkpoint we'll actually use:
# L4/A100 (>=20GB) -> 13B, T4 (16GB) -> 2B. On a T4 this skips copying the
# 15.7GB 13B file (huge time save); on L4 it skips the 4.5GB 2B.
try:
    import torch
    VRAM_GB = (
        torch.cuda.get_device_properties(0).total_memory / 1e9
        if torch.cuda.is_available() else 0
    )
except Exception:
    VRAM_GB = 0
USE_13B = VRAM_GB >= 20
WANTED_LTX = (
    'ltxv-13b-0.9.8-distilled-fp8.safetensors' if USE_13B
    else 'ltxv-2b-0.9.8-distilled-fp8.safetensors'
)
print(f'GPU VRAM ~{VRAM_GB:.0f}GB -> chỉ copy LTX: {WANTED_LTX}\n')

for subdir in ['diffusion_models', 'clip', 'vae', 'checkpoints']:
    drive_dir = DRIVE_ROOT / subdir
    comfy_dir = COMFY_MODELS / subdir
    comfy_dir.mkdir(parents=True, exist_ok=True)
    if not drive_dir.exists():
        print(f'[WARN] {drive_dir} not found, skip')
        continue
    for f in drive_dir.iterdir():
        if not f.is_file() or f.stat().st_size == 0:
            continue
        # Skip the LTX checkpoint that doesn't match this GPU.
        if subdir == 'checkpoints' and f.name.startswith('ltxv-') and f.name != WANTED_LTX:
            print(f'[SKIP gpu] {subdir}/{f.name} (không dùng trên GPU này)')
            continue
        dst = comfy_dir / f.name

        # Remove stale symlinks (from older notebook versions).
        if dst.is_symlink():
            dst.unlink()

        # Skip if real file already copied at correct size.
        if dst.exists() and not dst.is_symlink():
            if abs(dst.stat().st_size - f.stat().st_size) < 1024 * 1024:
                print(f'[SKIP] {subdir}/{f.name} ({dst.stat().st_size/(1024*1024):.0f}MB) already copied')
                continue
            dst.unlink()

        size_gb = f.stat().st_size / 1e9
        print(f'[COPY] {subdir}/{f.name} ({size_gb:.2f}GB) ...', end=' ', flush=True)
        shutil.copy2(f, dst)
        print('done')

print('\nAll models on /content/ComfyUI/models (mmap-friendly).')

## Bước 5: Cài cloudflared

Cloudflared quick tunnel cho HTTPS URL public, hỗ trợ cả HTTP và WebSocket. URL đổi mỗi session — paste lại vào BE `.env` mỗi lần.

In [ ]:
import os
import subprocess

if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run(
        [
            'wget', '-q',
            'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
            '-O', '/usr/local/bin/cloudflared',
        ],
        check=True,
    )
    os.chmod('/usr/local/bin/cloudflared', 0o755)
    print('[OK] cloudflared installed')
else:
    print('[SKIP] cloudflared already installed')

## Bước 6: Start ComfyUI in background

Wait đến khi ComfyUI respond `/system_stats` thì coi như ready (~30-60s cho lần đầu, ~20s cho subsequent).

In [ ]:
import subprocess
import time
import urllib.request

COMFY_LOG = '/content/comfyui.log'
_log_fp = open(COMFY_LOG, 'w')

comfy_proc = subprocess.Popen(
    ['python', '/content/ComfyUI/main.py', '--listen', '0.0.0.0', '--port', '8188'],
    stdout=_log_fp, stderr=subprocess.STDOUT,
    cwd='/content/ComfyUI',
)
print(f'ComfyUI starting (pid {comfy_proc.pid})... log at {COMFY_LOG}')

ready = False
for i in range(120):
    try:
        urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=2)
        ready = True
        break
    except Exception:
        time.sleep(1)

if not ready:
    print('[FAIL] ComfyUI did not respond in 120s. Last 30 log lines:')
    os.system(f'tail -30 {COMFY_LOG}')
else:
    print(f'[OK] ComfyUI ready after {i+1}s')

## Bước 7: Start cloudflared tunnel, lấy URL

Sau cell này, **copy URL `https://...trycloudflare.com`** in ra dưới và paste vào `backend/.env` ở máy local:

```
COMFY_BASE_URL=https://<id>.trycloudflare.com
```

Restart BE: `cd backend && run_dev.bat`.

In [ ]:
import subprocess
import re
import time

tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188', '--no-autoupdate'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
print('Cloudflared tunnel starting...')

tunnel_url = None
deadline = time.time() + 60
while time.time() < deadline:
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    m = re.search(r'(https://[a-z0-9-]+\.trycloudflare\.com)', line)
    if m:
        tunnel_url = m.group(1)
        break

if tunnel_url:
    print()
    print('=' * 72)
    print(f'COMFY_BASE_URL={tunnel_url}')
    print('=' * 72)
    print()
    print('Paste line trên (1 dòng) vào backend/.env, ghi đè COMFY_BASE_URL hiện có.')
    print('Sau đó restart BE: cd backend && run_dev.bat')
    print()
    print('URL chỉ tồn tại đến khi tunnel stop hoặc session Colab die.')
else:
    print('[FAIL] Không lấy được URL trong 60s. Xem output cloudflared phía trên.')

## Bước 8: Heartbeat — giữ session sống

Cell dưới chạy vô hạn, in heartbeat mỗi 10 phút. Đừng tắt cell này khi đang dùng ComfyUI từ BE local.

**Stop**: click nút stop ở cell — sẽ tự cleanup tunnel + ComfyUI.

In [ ]:
import time
from datetime import datetime, timezone

start = datetime.now(timezone.utc)
print(f'Heartbeat started at {start.strftime("%Y-%m-%d %H:%M:%S UTC")}')
print(f'Tunnel: {tunnel_url}')
print()

try:
    while True:
        time.sleep(600)
        elapsed_h = (datetime.now(timezone.utc) - start).total_seconds() / 3600
        now = datetime.now(timezone.utc).strftime('%H:%M:%S')
        print(f'[{now} UTC] Alive — {elapsed_h:.1f}h since start')
except KeyboardInterrupt:
    print('\nStopping...')
    tunnel_proc.terminate()
    comfy_proc.terminate()
    print('Tunnel + ComfyUI stopped. URL no longer valid.')